# Tutorial 53: Creation of Longitudinal Section - and Edge Dataframe

This Example demonstrates the capabilities of the class Dataframes_SIR3S_Model that extends SIR3S_Model be abilities to work directley with pandas dataframes. It is shown how to create dataframes containing information that does not concern individual elements types such as Nodes, Pipes, etc. but instead concerning more abstract SIR 3S data such as longitudinal sections or concatenations of multiple element types like hydraulic edges.

# Toolkit Release

In [1]:
#pip install 

# Imports

## SIR 3S Toolkit

### Regular Import/Init

In [2]:
SIR3S_SIRGRAF_DIR = r"C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2" #change to local path

In [3]:
from sir3stoolkit.core import wrapper

In [4]:
wrapper

<module 'sir3stoolkit.core.wrapper' from 'C:\\Users\\aUsername\\3S\\sir3stoolkit\\src\\sir3stoolkit\\core\\wrapper.py'>

In [5]:
wrapper.Initialize_Toolkit(SIR3S_SIRGRAF_DIR)

[2026-06-08 14:23:36,336] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 14:23:36,336] INFO in sir3stoolkit.core.wrapper: [Initialization] Using provided SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2
[2026-06-08 14:23:36,359] INFO in sir3stoolkit.core.wrapper: [Initialization] Initializing toolkit with SirGraf path: C:\3S\SIR 3S\SirGraf-90-15-00-24_Quebec-Upd2


### Additional Import/Init for Dataframes class

In [6]:
from sir3stoolkit.mantle.dataframes import SIR3S_Model_Dataframes

In [7]:
s3s = SIR3S_Model_Dataframes()

[2026-06-08 14:23:43,282] INFO in sir3stoolkit.core.wrapper: [Model Class Initialization] Initialization complete


## Additional

In [8]:
import pandas as pd
from shapely.geometry import Point
import re
import folium
from folium.plugins import HeatMap
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import contextily as cx

# Open Model

In [9]:
s3s.OpenModel(dbName=r"Toolkit_Tutorial53_Model.db3",
              providerType=s3s.ProviderTypes.SQLite,
              Mid="M-1-0-1",
              saveCurrentlyOpenModel=False,
              namedInstance="",
              userID="",
              password="")

[2026-06-08 14:23:50,849] INFO in sir3stoolkit.core.wrapper: Model is open for further operation


# Generate Non-Element Dataframes

## Edge dataframe

We can use the function [generate_edge_dataframe()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_edge_dataframe) to obtain a dataframe containing all edge objects in the model (each row representing one edge element). Below is a list of all element types, whose representatives that will be included in the dataframe, if present in the model.

In [10]:
edge_types = [
    'Pipe', 'Valve', 'SafetyValve', 'PressureRegulator', 'DifferentialRegulator',
    'FlapValve', 'PhaseSeparation', 'FlowControlUnit', 'ControlValve', 'Pump',
    'DistrictHeatingConsumer', 'DistrictHeatingFeeder', 'Compressor', 'HeaterCooler', 'HeatFeederConsumerStation'
]

HeatExchanger is not supported.

### Parametrization 1: Default

If we don't request any properties, we will only get [tk, Fkcont, geometry, fkKI, fkKK, element_type, L] as df columns.

In [11]:
df, timestamp_to_tuple_index = s3s.generate_edge_dataframe()
df.head(5)

[2026-06-08 14:23:50,877] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating edge dataframe ...
[2026-06-08 14:23:50,955] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-06-08 14:23:50,966] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-06-08 14:23:50,968] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 14:23:50,969] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 1 model_data properties.
[2026-06-08 14:23:50,970] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Fkcont'], geometry, end nodes...
[2026-06-08 14:23:51,179] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-06-08 14:23:51,220] INFO in sir3stoolkit.mantle.dataframes: [model_d

,Fkcont,L,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0,Pipe,5669301360686511351,5397948523091900401,"LINESTRING (713620.268 5578828.419, 713602.295...",5442010239090746007
1,5029128874972463118,0,Pipe,5397948523091900401,5239335112004772156,"LINESTRING (713602.295 5578860.106, 713574.062...",4917786378639043296
2,5029128874972463118,0,Pipe,5239335112004772156,5298886695042021307,"LINESTRING (713574.062 5578909.873, 713553.84 ...",4762482310382009633
3,5029128874972463118,0,Pipe,5298886695042021307,4993257270457791438,"LINESTRING (713553.84 5578945.533, 713553.394 ...",4987229536643024523
4,5029128874972463118,0,Pipe,4993257270457791438,5317865645994989592,"LINESTRING (713553.394 5578952.352, 713556.294...",5722206630503885118


### Parametrization 2: Properties + tk filter

We can request additional properties. Here we can mix model data and result properties in the same param. 

If a property is not defined for all edge element types, a column for that property will be created, but only populated, where values exists.

We can also filter for certain tks to exclusivly include in the dataframe. This is especially useful for large networks with known tks of interest. 

In [12]:
tks_of_interest = df["tk"][520:550].tolist() # now we just choose some randomly

In [13]:
df, timestamp_to_tuple_index = s3s.generate_edge_dataframe(properties=["Hal", "Kvr", "JV", "PVEC", "MVEC"],
                                                           tks=tks_of_interest)
df.head(5)

[2026-06-08 14:23:53,063] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating edge dataframe ...
[2026-06-08 14:23:53,196] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] TK filtering active. Requested tks: ['4957957828740199285', '5499776682108862751', '5285235772707481262', '5373158549695106826', '5078909058477631059', '5120684848284658605', '5652541284139882253', '5240450756436834634', '5257205059759497594', '4702663206643375451', '5217273986390542544', '5514879539114546017', '5586353371153335311', '5397401198360947165', '5684594766069449552', '4817917631249014225', '5110715607374589471', '5760916286273932524', '4656308032396859575', '5374020919990423089', '5661833458038115239', '4743997951091160959', '5014209100699808035', '4627580049017248376', '5018070164989726059', '5668240250035163192', '4681770111719926437', '5048206626643560289', '4874631416147311583', '4891291656447821692']
[2026-06-08 14:23:53,200] INFO in sir3stoolkit.mantle.dataframes: [edge dataf

,Fkcont,Hal,JV,Kvr,L,MVEC_end,MVEC_sequence,MVEC_start,PVEC_end,PVEC_sequence,PVEC_start,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0.0,"(0.0,)",2.0,0,"(0.0,)","((0.0, 0.0, 0.0),)","(0.0,)","(1.38355,)","((1.638605, 1.51108, 1.38355),)","(1.638605,)",Pipe,4863932769493405228,5295548655235318963,"LINESTRING (714408.376 5578937.983, 714404.59 ...",4957957828740199285
1,5029128874972463118,0.0,"(0.7570621,)",1.0,0,"(8.407589,)","((8.407589, 8.407589),)","(8.407589,)","(4.845395,)","((4.839625, 4.845395),)","(4.839625,)",Pipe,5172212715463480789,5392476601138952594,"LINESTRING (713580.476 5578511.084, 713584.785...",5285235772707481262
2,5029128874972463118,0.0,"(0.7657993,)",2.0,0,"(-8.407588,)","((-8.407588, -8.407588),)","(-8.407588,)","(3.997365,)","((3.981985, 3.997365),)","(3.981985,)",Pipe,5093129494518007690,5337935179219974656,"LINESTRING (713582.476 5578511.084, 713586.785...",5373158549695106826
3,5029128874972463118,0.0,"(0.0,)",2.0,0,"(0.0,)","((0.0, 0.0, 0.0, 0.0),)","(0.0,)","(1.06867,)","((1.38355, 1.27859, 1.17363, 1.06867),)","(1.38355,)",Pipe,5295548655235318963,5475286394948521111,"LINESTRING (714404.59 5578954.91, 714397.405 5...",5499776682108862751
4,5027846505677995694,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,Valve,5160850648779674898,4945502875946918021,POINT (190 208),5078909058477631059


The resulting df now includes model data properties such as "Hal" or "Kvr", but also result properties such as "JV" or result vector properties along pipe interior points like "PVEC" or "MVEC", which are split into the columns PVEC_start, PVEC_end, and PVEC_sequence = (PVEC_start, PVEC_2, PVEC_3, ..., PVEC_end).

As we didn't specify a timestamp, the result values are obtained for the stationary timestamp (s3s.GetTimeStamps()[1]).

Note that typical int cols like "Kvr" are float here. This happens due to the merge of many dataframes, some with NaN values. 

In [14]:
len(df)

30

We only get the rows for the requested tks.

### Parametrization 3: Properties + Timestamps

In [15]:
simulation_timestamps = s3s.GetTimeStamps()[0]

In [16]:
simulation_timestamps

['2023-02-13 00:00:00.000 +01:00',
 '2023-02-13 01:00:00.000 +01:00',
 '2023-02-13 02:00:00.000 +01:00',
 '2023-02-13 03:00:00.000 +01:00',
 '2023-02-13 04:00:00.000 +01:00',
 '2023-02-13 05:00:00.000 +01:00',
 '2023-02-13 06:00:00.000 +01:00',
 '2023-02-13 07:00:00.000 +01:00',
 '2023-02-13 08:00:00.000 +01:00',
 '2023-02-13 09:00:00.000 +01:00',
 '2023-02-13 10:00:00.000 +01:00',
 '2023-02-13 11:00:00.000 +01:00',
 '2023-02-13 12:00:00.000 +01:00',
 '2023-02-13 13:00:00.000 +01:00',
 '2023-02-13 14:00:00.000 +01:00',
 '2023-02-13 15:00:00.000 +01:00',
 '2023-02-13 16:00:00.000 +01:00',
 '2023-02-13 17:00:00.000 +01:00',
 '2023-02-13 18:00:00.000 +01:00',
 '2023-02-13 19:00:00.000 +01:00',
 '2023-02-13 20:00:00.000 +01:00',
 '2023-02-13 21:00:00.000 +01:00',
 '2023-02-13 22:00:00.000 +01:00',
 '2023-02-13 23:00:00.000 +01:00',
 '2023-02-14 00:00:00.000 +01:00']

We can request a list of timestamps for retrival of results properties. Therefore we take a slice from the available simulation timestamps.

In [17]:
df_edge, timestamp_to_tuple_index = s3s.generate_edge_dataframe(properties=["Hal", "Kvr", "JV", "PVEC", "MVEC"]
                                                                ,timestamps=[0, 5, -1])
df_edge.head(5)

[2026-06-08 14:23:54,931] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating edge dataframe ...
[2026-06-08 14:23:54,938] INFO in sir3stoolkit.mantle.dataframes: [edge dataframe] Generating dataframe for ObjectTypes.Pipe ...
[2026-06-08 14:23:54,945] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.Pipe
[2026-06-08 14:23:54,951] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 524 element(s) of element type ObjectTypes.Pipe.
[2026-06-08 14:23:54,954] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 3 model_data properties.
[2026-06-08 14:23:54,958] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Hal', 'Kvr', 'Fkcont'], geometry, end nodes...
[2026-06-08 14:23:55,343] INFO in sir3stoolkit.mantle.dataframes: [model_data] 2 non-empty end node columns were created.
[2026-06-08 14:23:55,400] INFO in sir3stoolkit.mantle.datafr

,Fkcont,Hal,JV,Kvr,L,MVEC_end,MVEC_sequence,MVEC_start,PVEC_end,PVEC_sequence,PVEC_start,element type,fkKI,fkKK,geometry,tk
0,5029128874972463118,0.0,"(4.303457e-23, 6.974091e-22, 0.0)",2.0,0,"(2.837623e-10, 1.142325e-09, 0.0)","((2.837623e-10, 2.837623e-10), (1.142325e-09, ...","(2.837623e-10, 1.142325e-09, 0.0)","(1.68863, 1.68863, 1.68864)","((1.615055, 1.68863), (1.615055, 1.68863), (1....","(1.615055, 1.615055, 1.61507)",Pipe,4730066059089961857,4917189080965035120,"LINESTRING (714262.483 5578857.42, 714269.543 ...",4614463970292122863
1,5029128874972463118,0.0,"(4.756155e-23, 0.0, 4.756155e-23)",1.0,0,"(-2.983143e-10, 0.0, -2.983143e-10)","((-2.983143e-10, -2.983143e-10, -2.983143e-10,...","(-2.983143e-10, 0.0, -2.983143e-10)","(3.59, 3.74699, 3.604835)","((3.304535, 3.345315, 3.386095, 3.426875, 3.46...","(3.304535, 3.461525, 3.31937)",Pipe,5129584372458662150,5332825919690090061,"LINESTRING (713738.297 5579219.902, 713793.23 ...",4615723899944629797
2,5029128874972463118,0.0,"(0.2111366, 0.4887606, 0.2371578)",2.0,0,"(-4.251046, -6.639488, -4.5234)","((-4.251046, -4.251046), (-6.639488, -6.639488...","(-4.251046, -6.639488, -4.5234)","(2.19852, 2.56674, 2.2329)","((2.210225, 2.19852), (2.577345, 2.56674), (2....","(2.210225, 2.577345, 2.244505)",Pipe,5070795580168283912,5725848577942138606,"LINESTRING (713650.613 5578990.488, 713649.498...",4621030304810285220
3,5029128874972463118,0.0,"(0.1849831, 0.4417774, 0.208755)",1.0,0,"(-24.13726, -37.92842, -25.70856)","((-24.13726, -24.13726, -24.13726, -24.13726),...","(-24.13726, -37.92842, -25.70856)","(4.21452, 4.61464, 4.251935)","((4.1376, 4.16324, 4.18888, 4.21452), (4.53218...","(4.1376, 4.532185, 4.174505)",Pipe,5106194195554624313,5416743601805578486,"LINESTRING (713283.932 5578718.123, 713299.538...",4621904482639719098
4,5029128874972463118,0.0,"(1.64011e-23, 7.243163e-24, 1.769575e-24)",2.0,0,"(1.737135e-10, -1.164153e-10, 5.729817e-11)","((1.737135e-10, 1.737135e-10, 1.737135e-10, 1....","(1.737135e-10, -1.164153e-10, 5.729817e-11)","(2.328365, 2.56135, 2.350165)","((2.304045, 2.30891, 2.313775, 2.318635, 2.323...","(2.304045, 2.53683, 2.325845)",Pipe,5395214589547810761,5186627743244018598,"LINESTRING (713231.461 5579074.621, 713266.467...",4623301797443553869


In the resulting dataframe every result value becomes a tuple, each element corresponding to one timestamp.

In [18]:
df_edge["JV"].iloc[0]

(4.303457e-23, 6.974091e-22, 0.0)

We also get a dict returned, that maps timestamp to tuple index.

In [19]:
timestamp_to_tuple_index

{'2023-02-13 00:00:00.000 +01:00': 0,
 '2023-02-13 05:00:00.000 +01:00': 1,
 '2023-02-14 00:00:00.000 +01:00': 2}

In [20]:
df_edge["JV"].iloc[0][timestamp_to_tuple_index[simulation_timestamps[0]]]

4.303457e-23

We can also get series of the "JV" values for one timestamp in the different pipes.

In [28]:
df_edge["JV"].str[timestamp_to_tuple_index[simulation_timestamps[0]]]

0      4.303457e-23
1      4.756155e-23
2      2.111366e-01
3      1.849831e-01
4      1.640110e-23
5      4.420876e-24
6      8.080109e-02
7      7.522397e-23
8      6.953945e-20
9      4.121174e-01
10     4.974777e-01
11     9.559039e-23
12     3.172845e-01
13     1.986323e-23
14     5.729455e-25
15     0.000000e+00
16     1.619071e-22
17     4.238189e-23
18     4.253159e-01
19     1.903912e-24
20     1.195405e-22
21     1.472177e-02
22     1.839792e-23
23     2.993693e-01
24     4.505997e-01
25     0.000000e+00
26     1.527271e-01
27     2.120296e-23
28     1.021399e-23
29     1.308296e-22
30     0.000000e+00
31     2.925582e-01
32     4.394961e-01
33     7.883015e-01
34     7.355869e-20
35     6.155628e-24
36     2.288080e-25
37     1.548829e-01
38     1.925734e-24
39     0.000000e+00
40     1.548817e-01
41     2.858229e-01
42     1.912648e-23
43     1.562514e-23
44     3.565702e-23
45     1.158490e-01
46     1.110624e-23
47     1.742017e-01
48     7.028960e-02
49     0.000000e+00


For already vectorized results (pipe interior points) we get tuple of tuples with the time tuple being the higher level one.

In [22]:
df_edge["MVEC_sequence"].iloc[0]

((2.837623e-10, 2.837623e-10), (1.142325e-09, 1.142325e-09), (0.0, 0.0))

In [23]:
df_edge["MVEC_sequence"].iloc[0][timestamp_to_tuple_index[simulation_timestamps[0]]]

(2.837623e-10, 2.837623e-10)

## AGSN (Longitudinal Sections)

We can use the method [generate_longitudinal_section_dataframes()](https://3sconsult.github.io/sir3stoolkit/references/sir3stoolkit.mantle.html#sir3stoolkit.mantle.dataframes.SIR3S_Model_Dataframes.generate_longitudinal_section_dataframes) to obtain a list indexed by (lfdnr-1) filled with size-2-tuples with 0 - Supply and 1 - Return..

In [24]:
dfs = s3s.generate_longitudinal_section_dataframes()

[2026-06-08 14:24:05,208] INFO in sir3stoolkit.mantle.dataframes: [model_data] Generating model_data dataframe for element type: ObjectTypes.AGSN_HydraulicProfile
[2026-06-08 14:24:05,213] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieved 3 element(s) of element type ObjectTypes.AGSN_HydraulicProfile.
[2026-06-08 14:24:05,217] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] No properties given → using ALL model_data properties for ObjectTypes.AGSN_HydraulicProfile.
[2026-06-08 14:24:05,218] INFO in sir3stoolkit.mantle.dataframes: [Resolving model_data Properties] Using 9 model_data properties.
[2026-06-08 14:24:05,220] INFO in sir3stoolkit.mantle.dataframes: [model_data] Retrieving model_data properties ['Name', 'Lfdnr', 'Aktiv', 'AllNodesAndLinks', 'ObjsString', 'MainWay', 'Tk', 'Pk', 'InVariant']...
[2026-06-08 14:24:05,259] INFO in sir3stoolkit.mantle.dataframes: [model_data] Done. Shape: (3, 10)
[2026-06-08 14:24:05,263] INFO in sir3stoolkit.m

In [25]:
dfs[0][1].head() # Lfdnr 1 RL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,MVEC_sequence,MVEC_start,PDAMPF,PHR,PMIN,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMAX_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,PVECMIN_INST_start,PVEC_end,PVEC_sequence,PVEC_start,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_end,RHOVEC_sequence,RHOVEC_start,SVEC_end,SVEC_sequence,SVEC_start,TI,TK,TTRVEC_end,TTRVEC_sequence,TTRVEC_start,TVEC_end,TVEC_sequence,TVEC_start,VAV,VI,VK,VOLDA,WVL,ZVEC_end,ZVEC_sequence,ZVEC_start,l_sum,AGSN_Lfdnr,AGSN_Name
0,5025945677694931826,Rohr R-E0 R-K4163S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5691533564979419761,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5025945677694931826,5025945677694931826,False,713621.921383,5.578219e+06,False,5025945677694931826,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713621.921 5578218.954, 713616.649...",5160850648779674898,4851678476955084637,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.004934843,)","(26.63305,)","(0.4088626,)","(0.0,)","(0.0,)","(0.1548843,)","(-79.74764,)","((-79.74764, -79.74764, -79.74764),)","(-79.74764,)","(0.1976851,)","(0.002377737,)","(3.953543,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(-287.0915,)","(-287.0915,)","(-287.0915,)","(983.7845,)","(983.7839,)","(983.7839,)","((983.7845, 983.7842, 983.7839),)","(983.7845,)","(15.3517,)","((0.0, 7.67585, 15.3517),)","(0.0,)","(59.83105,)","(59.83229,)","(0.311801,)","((0.3167359, 0.3142685, 0.311801),)","(0.3167359,)","(59.83231,)","((59.83105, 59.83167, 59.83231),)","(59.83105,)","(-0.8641331,)","(-0.8641329,)","(-0.8641334,)","(0.0,)","(0.0,)","(542.2,)","((541.43, 541.815, 542.2),)","(541.43,)",15.351701,1,Längsschnitt
1,5216742060270992761,Rohr R-K4163S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5048873293262650113,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5216742060270992761,5216742060270992761,False,713616.648712,5.578233e+06,False,5216742060270992761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713616.649 5578233.372, 713616.465...",4851678476955084637,5469753985021987570,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.004021031,)","(27.238,)","(0.3407188,)","(0.0,)","(0.0,)","(0.1548841,)","(-79.74764,)","((-79.74764, -79.74764, -79.74764),)","(-79.74764,)","(0.1976953,)","(0.001937438,)","(3.885054,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(-287.0915,)","(-287.0915,)","(-287.0915,)","(983.7839,)","(983.7833,)","(983.7833,)","((983.7839, 983.7836, 983.7833),)","(983.7839,)","(12.50895,)","((0.0, 6.254475, 12.50895),)","(0.0,)","(59.83229,)","(59.8333,)","(0.30778,)","((0.311801, 0.3097906, 0.30778),)","(0.311801,)","(59.83331,)","((59.83231, 59.83279, 59.83331),)","(59.8

In [26]:
dfs[1][0].head() # Lfdnr 2 VL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,MVEC_sequence,MVEC_start,PDAMPF,PHR,PMIN,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMAX_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,PVECMIN_INST_start,PVEC_end,PVEC_sequence,PVEC_start,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_end,RHOVEC_sequence,RHOVEC_start,SVEC_end,SVEC_sequence,SVEC_start,TI,TK,TTRVEC_end,TTRVEC_sequence,TTRVEC_start,TVEC_end,TVEC_sequence,TVEC_start,VAV,VI,VK,VOLDA,WVL,ZVEC_end,ZVEC_sequence,ZVEC_start,l_sum,AGSN_Lfdnr,AGSN_Name
0,5691533564979419761,Rohr V-E0 V-K1683S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5025945677694931826,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5691533564979419761,5691533564979419761,False,713619.921383,5.578219e+06,False,5691533564979419761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713619.921 5578218.954, 713614.649...",5398100694284104779,4825391580467484032,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.004844132,)","(43.12018,)","(0.661968,)","(0.0,)","(0.0,)","(0.1527268,)","(79.74764,)","((79.74764, 79.74764, 79.74764),)","(79.74764,)","(0.701074,)","(0.002344616,)","(5.800529,)","(5.80053,)","((5.878635, 5.83958, 5.80053),)","(5.878635,)","(5.80053,)","((5.878635, 5.83958, 5.80053),)","(5.878635,)","(5.80053,)","((5.878635, 5.83958, 5.80053),)","(5.878635,)","(287.0915,)","(287.0915,)","(287.0915,)","(965.7,)","(965.7012,)","(965.7012,)","((965.7, 965.7006, 965.7012),)","(965.7,)","(15.3517,)","((0.0, 7.67585, 15.3517),)","(0.0,)","(89.99999,)","(89.99802,)","(0.004844132,)","((0.0, 0.002422066, 0.004844132),)","(0.0,)","(89.99802,)","((90.0, 89.99902, 89.99802),)","(90.0,)","(0.8803148,)","(0.8803153,)","(0.8803143,)","(0.0,)","(10082.88,)","(542.29,)","((541.49, 541.89, 542.29),)","(541.49,)",15.351701,2,Längsschnitt in mlc
1,5048873293262650113,Rohr V-K1683S V-K1693S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5216742060270992761,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,1,0.005,0,350,345.6,Vorlauf,,5048873293262650113,5048873293262650113,False,713614.648712,5.578233e+06,False,5048873293262650113,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713614.649 5578233.372, 713614.465...",4825391580467484032,5180617780362861593,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.003947125,)","(43.58081,)","(0.5451502,)","(0.0,)","(0.0,)","(0.1527269,)","(79.74764,)","((79.74764, 79.74764, 79.74764),)","(79.74764,)","(0.7010268,)","(0.001910453,)","(5.732326,)","(5.732325,)","((5.80053, 5.76643, 5.732325),)","(5.80053,)","(5.732325,)","((5.80053, 5.76643, 5.732325),)","(5.80053,)","(5.732325,)","((5.80053, 5.76643, 5.732325),)","(5.80053,)","(287.0915,)","(287.0915,)","(287.0915,)","(965.7012,)","(965.7021,)","(965.7021,)","((965.7012, 965.7017, 965.7021),)","(965.7012,)","(12.50895,)","((0.0, 6.254475, 12.50895),)","(0.0,)","(89.99802,)","(89.9964,)","(0.008791257,)","((0.004844132, 0.006817694, 0.008791257),)","(0.004844132,)","(89.9964,)","((89.99802, 89.99722, 89.9964),)","(89.99802,)","

In [27]:
dfs[1][1].head() # Lfdnr 2 RL

,tk,Name,FkdtroRowd,Fkltgr,Fkstrasse,L,Lzu,Rau,Jlambs,Lambda0,Zein,Zaus,Zuml,Asoll,Indschall,Baujahr,Hal,Fkcont,Fk2lrohr,Beschreibung,Idreferenz,Iplanung,Kvr,LineWidthMM,DottedLine,DN,Di,KvrKlartext,HasClosedNSCHs,Tk,Pk,InVariant,Xkor,Ykor,GeometriesDiffer,bz.Fk,bz.Qsvb,bz.Irtrenn,bz.Leckstatus,bz.Leckstart,bz.Leckend,bz.Leckort,bz.Leckmenge,bz.Imptnz,bz.Zvlimptnz,bz.Kantenzv,bz.ITrennWithNSCH,geometry,fkKI,fkKK,PipeTable: Table Name,PipeTable: Name,PipeTable: Fk,PipeTable: Dn,PipeTable: Di,PipeTable: Da,PipeTable: S,PipeTable: Wsteig,PipeTable: Wtiefe,PipeTable: Kt,PipeTable: Pn,PipeTable: Ausfallzeit,PipeTable: Reparatur,PipeTable: Rehabilitation,PipeTable: Tk,PipeTable: Pk,PipeTable: InVariant,A,DTTR,DWVERL,DWVERLABS,IAKTIV,IRTRENN,JV,MVEC_end,MVEC_sequence,MVEC_start,PDAMPF,PHR,PMIN,PVECMAX_INST_end,PVECMAX_INST_sequence,PVECMAX_INST_start,PVECMIN_INST_end,PVECMIN_INST_sequence,PVECMIN_INST_start,PVEC_end,PVEC_sequence,PVEC_start,QMAV,QMI,QMK,RHOI,RHOK,RHOVEC_end,RHOVEC_sequence,RHOVEC_start,SVEC_end,SVEC_sequence,SVEC_start,TI,TK,TTRVEC_end,TTRVEC_sequence,TTRVEC_start,TVEC_end,TVEC_sequence,TVEC_start,VAV,VI,VK,VOLDA,WVL,ZVEC_end,ZVEC_sequence,ZVEC_start,l_sum,AGSN_Lfdnr,AGSN_Name
0,5025945677694931826,Rohr R-E0 R-K4163S,4816511167455310973,4779752876656844188,5204829332281547346,15.351700,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5691533564979419761,OSM: Knoten 476971238 -> Knoten 299394923; Län...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5025945677694931826,5025945677694931826,False,713621.921383,5.578219e+06,False,5025945677694931826,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713621.921 5578218.954, 713616.649...",5160850648779674898,4851678476955084637,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.004934843,)","(26.63305,)","(0.4088626,)","(0.0,)","(0.0,)","(0.1548843,)","(-79.74764,)","((-79.74764, -79.74764, -79.74764),)","(-79.74764,)","(0.1976851,)","(0.002377737,)","(3.953543,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(3.95354,)","((4.025455, 3.9895, 3.95354),)","(4.025455,)","(-287.0915,)","(-287.0915,)","(-287.0915,)","(983.7845,)","(983.7839,)","(983.7839,)","((983.7845, 983.7842, 983.7839),)","(983.7845,)","(15.3517,)","((0.0, 7.67585, 15.3517),)","(0.0,)","(59.83105,)","(59.83229,)","(0.311801,)","((0.3167359, 0.3142685, 0.311801),)","(0.3167359,)","(59.83231,)","((59.83105, 59.83167, 59.83231),)","(59.83105,)","(-0.8641331,)","(-0.8641329,)","(-0.8641334,)","(0.0,)","(0.0,)","(542.2,)","((541.43, 541.815, 542.2),)","(541.43,)",15.351701,2,Längsschnitt in mlc
1,5216742060270992761,Rohr R-K4163S R-K4173S,4816511167455310973,4779752876656844188,5204829332281547346,12.508950,0,0.05,1,0,0,0,0,1000,0,,0,5029128874972463118,5048873293262650113,OSM: Knoten 299394923 -> Knoten 4105649557; Lä...,39785520,0,2,0.005,0,350,345.6,Rücklauf,,5216742060270992761,5216742060270992761,False,713616.648712,5.578233e+06,False,5216742060270992761,0,0,0,0,0,0,0,0,0,0,0,"LINESTRING (713616.649 5578233.372, 713616.465...",4851678476955084637,5469753985021987570,KMR,DN 350,4927972999088213410,350,345.6,355.6,5.0,0,0,0.54,0,0,0,1300,4816511167455310973,4816511167455310973,False,"(0.0,)","(0.004021031,)","(27.238,)","(0.3407188,)","(0.0,)","(0.0,)","(0.1548841,)","(-79.74764,)","((-79.74764, -79.74764, -79.74764),)","(-79.74764,)","(0.1976953,)","(0.001937438,)","(3.885054,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(3.885055,)","((3.95354, 3.9193, 3.885055),)","(3.95354,)","(-287.0915,)","(-287.0915,)","(-287.0915,)","(983.7839,)","(983.7833,)","(983.7833,)","((983.7839, 983.7836, 983.7833),)","(983.7839,)","(12.50895,)","((0.0, 6.254475, 12.50895),)","(0.0,)","(59.83229,)","(59.8333,)","(0.30778,)","((0.311801, 0.3097906, 0.30778),)","(0.311801,)","(59.83331,)","((59.83231, 59.83279, 59.83331),)"